In [1]:
import sys
sys.path.append('../..')

import pandas as pd
import networkx as nx
import itertools

from networks.utils.get_monthly_nodes import get_monthly_nodes
# from networks.utils.calculate_similarity import calculate_similarity
from networks.utils.calculate_better_similarity import calculate_similarity
from networks.utils.prune_to_average_degree import prune_to_average_degree

In [2]:
from itertools import combinations
import numpy as np
from dtaidistance import dtw

In [3]:
city = "Cluj"
start = '1960-01'
end = '1990-12'

In [4]:
monthly_nodes = get_monthly_nodes(city, start, end)
months = list(monthly_nodes.keys())

In [5]:
dtw_distances = []

for m1, m2 in combinations(months, 2):
    d1 = np.array(monthly_nodes[m1]['tg_derivatives'], dtype=np.double)
    d2 = np.array(monthly_nodes[m2]['tg_derivatives'], dtype=np.double)

    # z-normalize the sequences (important)
    d1 = (d1 - d1.mean()) / d1.std()
    d2 = (d2 - d2.mean()) / d2.std()

    dist = dtw.distance(d1, d2)
    dtw_distances.append(dist)

In [6]:
dtw_mean = np.mean(dtw_distances)
dtw_std  = np.std(dtw_distances)

In [7]:
tg_diffs, tn_diffs, tx_diffs = [], [], []

for m1, m2 in combinations(months, 2):
    tg_diffs.append(abs(monthly_nodes[m1]['mean_tg'] - monthly_nodes[m2]['mean_tg']))
    tn_diffs.append(abs(monthly_nodes[m1]['mean_tn'] - monthly_nodes[m2]['mean_tn']))
    tx_diffs.append(abs(monthly_nodes[m1]['mean_tx'] - monthly_nodes[m2]['mean_tx']))

In [8]:
tg_mean, tg_std = np.mean(tg_diffs), np.std(tg_diffs)
tn_mean, tn_std = np.mean(tn_diffs), np.std(tn_diffs)
tx_mean, tx_std = np.mean(tx_diffs), np.std(tx_diffs)

In [9]:
stats = {
    'dtw_mean': dtw_mean,
    'dtw_std': dtw_std,
    'tg_mean': tg_mean,
    'tg_std': tg_std,
    'tn_mean': tn_mean,
    'tn_std': tn_std,
    'tx_mean': tx_mean,
    'tx_std': tx_std
}

In [10]:
# --- Get the monthly data ---
cluj_nodes = get_monthly_nodes(city, start, end)

# --- Network Construction ---
G = nx.Graph()
simple_G = nx.Graph()

# Add nodes to the graph
for month, data in cluj_nodes.items():
    G.add_node(month, **data)
    simple_G.add_node(month)

# --- Calculate similarities and add edges ---
months = list(cluj_nodes.keys())
similarity_threshold = 0.3

for month1, month2 in itertools.combinations(months, 2):
    month1_data = cluj_nodes[month1]
    month2_data = cluj_nodes[month2]
    
    similarity_score = calculate_similarity(month1_data, month2_data, stats)
    
    if similarity_score > similarity_threshold:
        G.add_edge(month1, month2, weight=similarity_score)
        simple_G.add_edge(month1, month2, weight = similarity_score)

# --- Inspect the graph ---
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")

Number of nodes: 372
Number of edges: 56449


In [ ]:
g_type = 'global'

graph_path = f"{city}_{g_type}_{start}_{end}_sim_{int(similarity_threshold*100)}.graphml"
nx.write_graphml(simple_G, graph_path)


## prune to avg degree 64

In [12]:
pruned_graph = prune_to_average_degree(graph_path, target_avg_degree=64)
nx.write_graphml(simple_G, graph_path)

Pruning 44545 edges to reach average degree of 64...
Finished. Final Average Degree: 64.00


# give month and year

In [ ]:
import networkx as nx

# G = nx.read_graphml(graph_path)
G = pruned_graph

for node, data in G.nodes(data=True):
    try:
        year, month = node.split("-")
        data["year"] = int(year)
        data["month"] = int(month)
    except ValueError:
        data["year"] = None
        data["month"] = None

pruned_YM_path = f"{city}_{g_type}_pruned_{start}_{end}.graphml"

nx.write_graphml(G, pruned_YM_path)

# END

In [14]:
# # --- Get the monthly data ---
# cluj_nodes = get_monthly_nodes('Cluj')

# # --- Network Construction ---
# G = nx.Graph()
# simple_G = nx.Graph()

# # Add nodes to the graph
# for month, data in cluj_nodes.items():
#     G.add_node(month, **data)
#     simple_G.add_node(month)

# # --- Calculate similarities and add edges ---
# months = list(cluj_nodes.keys())
# similarity_threshold = 0.5

# for month1, month2 in itertools.combinations(months, 2):
#     month1_data = cluj_nodes[month1]
#     month2_data = cluj_nodes[month2]
    
#     similarity_score = calculate_similarity(month1_data, month2_data, stats, weights={'deriv': 0.1, 'tg': 0.5, 'tn': 0.2, 'tx': 0.2})
    
#     if similarity_score > similarity_threshold:
#         G.add_edge(month1, month2, weight=similarity_score)
#         simple_G.add_edge(month1, month2, weight = similarity_score)

# # --- Inspect the graph ---
# print(f"Number of nodes: {G.number_of_nodes()}")
# print(f"Number of edges: {G.number_of_edges()}")

In [15]:
# nx.write_graphml(simple_G, "graph_fin_sim_20_1.graphml")

# Removing weak links


In [16]:
# import networkx as nx

# # Load the GraphML graph
# G = nx.read_graphml("graph_fin_sim_20_1.graphml")

# # Create an empty graph of the same type
# G_filtered = G.__class__()
# G_filtered.graph.update(G.graph)

# # Copy nodes (with attributes)
# G_filtered.add_nodes_from(G.nodes(data=True))

# # Filter edges by weight
# for u, v, data in G.edges(data=True):
#     # GraphML usually stores attributes as strings
#     weight = float(data.get("weight", 0))
    
#     if weight > 0.75:
#         G_filtered.add_edge(u, v, **data)

# # Save the filtered graph (optional)
# nx.write_graphml(G_filtered, "graph_01_75.graphml")


# giving month and year

In [17]:
# import networkx as nx

# G = nx.read_graphml("graph_07_75.graphml")

# # for node, data in G.nodes(data=True):
# #     # node is a string like "1950-05"
# #     try:
# #         year, month = node.split("-")
# #         data["month"] = int(month)  # 1–12
# #     except ValueError:
# #         data["month"] = None

# for node, data in G.nodes(data=True):
#     try:
#         year, month = node.split("-")
#         data["year"] = int(year)
#         data["month"] = int(month)
#     except ValueError:
#         data["year"] = None
#         data["month"] = None

# nx.write_graphml(G, "graph_details_07_75.graphml")

# FULL ON HEAD TO TOE CREATION

In [1]:
import sys
sys.path.append('../..')

import pandas as pd
import networkx as nx
import itertools
import json
import os

from networks.utils.get_monthly_nodes import get_monthly_nodes
# from networks.utils.calculate_similarity import calculate_similarity
from networks.utils.calculate_better_similarity import calculate_similarity
from networks.utils.prune_to_average_degree import prune_to_average_degree

from itertools import combinations
import numpy as np
from dtaidistance import dtw

In [ ]:
### NOT THIS ####
def create_all_global_networks(specifications):
    # JSON file to store the thresholds
    json_filename = "network_thresholds.json"

    for city, start, end in specifications:

        print(f"--- Starting work on {city} : {start} - {end} ---")
        
        # 1. Fetch data (Only once)
        monthly_nodes = get_monthly_nodes(city, start, end)
        months = list(monthly_nodes.keys())

        # 2. Calculate Statistics (DTW and attribute differences)
        dtw_distances = []
        tg_diffs, tn_diffs, tx_diffs = [], [], []

        for m1, m2 in combinations(months, 2):
            # DTW Calculation
            d1 = np.array(monthly_nodes[m1]['tg_derivatives'], dtype=np.double)
            d2 = np.array(monthly_nodes[m2]['tg_derivatives'], dtype=np.double)

            # Z-normalize
            d1 = (d1 - d1.mean()) / d1.std()
            d2 = (d2 - d2.mean()) / d2.std()

            dtw_distances.append(dtw.distance(d1, d2))

            # Absolute differences
            tg_diffs.append(abs(monthly_nodes[m1]['mean_tg'] - monthly_nodes[m2]['mean_tg']))
            tn_diffs.append(abs(monthly_nodes[m1]['mean_tn'] - monthly_nodes[m2]['mean_tn']))
            tx_diffs.append(abs(monthly_nodes[m1]['mean_tx'] - monthly_nodes[m2]['mean_tx']))

        stats = {
            'dtw_mean': np.mean(dtw_distances),
            'dtw_std': np.std(dtw_distances),
            'tg_mean': np.mean(tg_diffs),
            'tg_std': np.std(tg_diffs),
            'tn_mean': np.mean(tn_diffs),
            'tn_std': np.std(tn_diffs),
            'tx_mean': np.mean(tx_diffs),
            'tx_std': np.std(tx_diffs)
        }

        print(f"Statistics calculated")

        # 3. Network Construction
        G = nx.Graph()
        # We only need one graph object to start with
        for month, data in monthly_nodes.items():
            G.add_node(month, **data)

        # 4. Calculate similarities and add edges
        similarity_threshold = 0.3
        
        for month1, month2 in itertools.combinations(months, 2):
            month1_data = monthly_nodes[month1]
            month2_data = monthly_nodes[month2]
            
            similarity_score = calculate_similarity(month1_data, month2_data, stats)
            
            if similarity_score > similarity_threshold:
                G.add_edge(month1, month2, weight=similarity_score)

        print(f"Initial Graph - Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

        # 5. Save Initial Graph and Prune
        g_type = 'global'
        graph_path = f"{city}_{g_type}_{start}_{end}_sim_{int(similarity_threshold*100)}.graphml"
        
        # Save the unpruned version first
        nx.write_graphml(G, graph_path)

        # Prune (assuming this function loads graph_path, prunes it, and returns the object)
        pruned_graph = prune_to_average_degree(graph_path, target_avg_degree=64)

        # ---------------------------------------------------------
        # NEW LOGIC: Extract Threshold and Save to JSON
        # ---------------------------------------------------------
        
        # The threshold is effectively the minimum weight remaining in the pruned graph
        if pruned_graph.number_of_edges() > 0:
            # Extract all weights
            weights = [d['weight'] for u, v, d in pruned_graph.edges(data=True)]
            # Find the minimum
            final_threshold = float(min(weights))
        else:
            final_threshold = 0.0

        # Prepare the key and data
        json_key = f"{city}_{start}_{end}"
        
        # Load existing JSON if it exists, otherwise create new dict
        if os.path.exists(json_filename):
            with open(json_filename, 'r') as f:
                try:
                    data_store = json.load(f)
                except json.JSONDecodeError:
                    data_store = {}
        else:
            data_store = {}

        # Update and write back
        data_store[json_key] = final_threshold
        
        with open(json_filename, 'w') as f:
            json.dump(data_store, f, indent=4)
            
        print(f"Threshold {final_threshold:.4f} saved for {json_key}")
        # ---------------------------------------------------------

        # 6. Post-Processing: Add Year/Month attributes
        for node, data in pruned_graph.nodes(data=True):
            try:
                # Ensure the node is string before split, depending on how it was loaded
                year, month = str(node).split("-")
                data["year"] = int(year)
                data["month"] = int(month)
            except ValueError:
                data["year"] = None
                data["month"] = None

        # 7. Save Final Pruned Graph
        pruned_YM_path = f"{city}_{g_type}_pruned_{start}_{end}.graphml"
        nx.write_graphml(pruned_graph, pruned_YM_path)

        print(f"Network pruned and saved to {pruned_YM_path}")
        print("-" * 30)

In [2]:
######## OLD ONE ###############################
def create_all_global_networks(specifications):

    json_filename = "network_thresholds.json"

    for city, start, end in specifications:

        print(f"Starting to work on {city} : {start} - {end}")
        
        monthly_nodes = get_monthly_nodes(city, start, end)
        months = list(monthly_nodes.keys())

        dtw_distances = []

        for m1, m2 in combinations(months, 2):
            d1 = np.array(monthly_nodes[m1]['tg_derivatives'], dtype=np.double)
            d2 = np.array(monthly_nodes[m2]['tg_derivatives'], dtype=np.double)

            # z-normalize the sequences (important)
            d1 = (d1 - d1.mean()) / d1.std()
            d2 = (d2 - d2.mean()) / d2.std()

            dist = dtw.distance(d1, d2)
            dtw_distances.append(dist)

        dtw_mean = np.mean(dtw_distances)
        dtw_std  = np.std(dtw_distances)

        tg_diffs, tn_diffs, tx_diffs = [], [], []

        for m1, m2 in combinations(months, 2):
            tg_diffs.append(abs(monthly_nodes[m1]['mean_tg'] - monthly_nodes[m2]['mean_tg']))
            tn_diffs.append(abs(monthly_nodes[m1]['mean_tn'] - monthly_nodes[m2]['mean_tn']))
            tx_diffs.append(abs(monthly_nodes[m1]['mean_tx'] - monthly_nodes[m2]['mean_tx']))

        tg_mean, tg_std = np.mean(tg_diffs), np.std(tg_diffs)
        tn_mean, tn_std = np.mean(tn_diffs), np.std(tn_diffs)
        tx_mean, tx_std = np.mean(tx_diffs), np.std(tx_diffs)

        stats = {
            'dtw_mean': dtw_mean,
            'dtw_std': dtw_std,
            'tg_mean': tg_mean,
            'tg_std': tg_std,
            'tn_mean': tn_mean,
            'tn_std': tn_std,
            'tx_mean': tx_mean,
            'tx_std': tx_std
        }

        print(f"Statistics calculated")

        # --- Get the monthly data ---
        monthly_nodes = get_monthly_nodes(city, start, end)

        # --- Network Construction ---
        G = nx.Graph()
        simple_G = nx.Graph()

        # Add nodes to the graph
        for month, data in monthly_nodes.items():
            G.add_node(month, **data)
            simple_G.add_node(month)

        # --- Calculate similarities and add edges ---
        months = list(monthly_nodes.keys())
        similarity_threshold = 0.3

        for month1, month2 in itertools.combinations(months, 2):
            month1_data = monthly_nodes[month1]
            month2_data = monthly_nodes[month2]
            
            similarity_score = calculate_similarity(month1_data, month2_data, stats)
            
            if similarity_score > similarity_threshold:
                G.add_edge(month1, month2, weight=similarity_score)
                simple_G.add_edge(month1, month2, weight = similarity_score)

        # --- Inspect the graph ---
        print(f"Number of nodes: {G.number_of_nodes()}")
        print(f"Number of edges: {G.number_of_edges()}")

        print(f"First network created")

        g_type = 'global'

        graph_path = f"{city}_{g_type}_{start}_{end}_sim_{int(similarity_threshold*100)}.graphml"
        nx.write_graphml(simple_G, graph_path)

        pruned_graph = prune_to_average_degree(graph_path, target_avg_degree=64)
        # nx.write_graphml(simple_G, graph_path)

        # ---------------------------------------------------------
        # NEW LOGIC: Extract Threshold and Save to JSON
        # ---------------------------------------------------------
        
        # The threshold is effectively the minimum weight remaining in the pruned graph
        if pruned_graph.number_of_edges() > 0:
            # Extract all weights
            weights = [d['weight'] for u, v, d in pruned_graph.edges(data=True)]
            # Find the minimum
            final_threshold = float(min(weights))
        else:
            final_threshold = 0.0

        # Prepare the key and data
        json_key = f"{city}_{start}_{end}"
        
        # Load existing JSON if it exists, otherwise create new dict
        if os.path.exists(json_filename):
            with open(json_filename, 'r') as f:
                try:
                    data_store = json.load(f)
                except json.JSONDecodeError:
                    data_store = {}
        else:
            data_store = {}

        # Update and write back
        data_store[json_key] = final_threshold
        
        with open(json_filename, 'w') as f:
            json.dump(data_store, f, indent=4)
            
        print(f"Threshold {final_threshold:.4f} saved for {json_key}")
        # ---------------------------------------------------------

        # G = nx.read_graphml(graph_path)
        G = pruned_graph

        for node, data in G.nodes(data=True):
            try:
                year, month = node.split("-")
                data["year"] = int(year)
                data["month"] = int(month)
            except ValueError:
                data["year"] = None
                data["month"] = None

        pruned_YM_path = f"{city}_{g_type}_pruned_{start}_{end}.graphml"

        nx.write_graphml(G, pruned_YM_path)

        print(f"Network pruned and saved")

In [3]:
specifications = [
    ('Cluj', '1960-01', '1990-12'),
    ('Cluj', '1991-01', '2024-12'),
    ('Cluj', '1960-01', '2024-12'),
    ('Bacskatopolya', '1960-01', '1990-12'),
    ('Bacskatopolya', '1991-01', '2024-12'),
    ('Bacskatopolya', '1960-01', '2024-12'),
    ('Brasov', '1960-01', '1990-12'),
    ('Brasov', '1991-01', '2024-12'),
    ('Brasov', '1960-01', '2024-12'),
    ('Deva', '1960-01', '1990-12'),
    ('Deva', '1991-01', '2024-12'),
    ('Deva', '1960-01', '2024-12'),
    ('Gheorgheni', '1960-01', '1990-12'),
    ('Gheorgheni', '1991-01', '2024-12'),
    ('Gheorgheni', '1960-01', '2024-12'),
    ('Gyor', '1960-01', '1990-12'),
    ('Gyor', '1991-01', '2024-12'),
    ('Gyor', '1960-01', '2024-12'),
    ('Kassa', '1960-01', '1990-12'),
    ('Kassa', '1991-01', '2024-12'),
    ('Kassa', '1960-01', '2024-12'),
    ('Kecskemet', '1960-01', '1990-12'),
    ('Kecskemet', '1991-01', '2024-12'),
    ('Kecskemet', '1960-01', '2024-12'),
    ('Keszthely', '1960-01', '1990-12'),
    ('Keszthely', '1991-01', '2024-12'),
    ('Keszthely', '1960-01', '2024-12'),
    ('Oradea', '1960-01', '1990-12'),
    ('Oradea', '1991-01', '2024-12'),
    ('Oradea', '1960-01', '2024-12'),
    ('Pecs', '1960-01', '1990-12'),
    ('Pecs', '1991-01', '2024-12'),
    ('Pecs', '1960-01', '2024-12'),
]

In [4]:
create_all_global_networks(specifications)

Starting to work on Cluj : 1960-01 - 1990-12
Statistics calculated
Number of nodes: 371
Number of edges: 56183
First network created
Pruning 44311 edges to reach average degree of 64...
Finished. Final Average Degree: 64.00
Threshold 0.6974 saved for Cluj_1960-01_1990-12
Network pruned and saved
Starting to work on Cluj : 1991-01 - 2024-12
Statistics calculated
Number of nodes: 407
Number of edges: 67338
First network created
Pruning 54314 edges to reach average degree of 64...
Finished. Final Average Degree: 64.00
Threshold 0.7045 saved for Cluj_1991-01_2024-12
Network pruned and saved
Starting to work on Cluj : 1960-01 - 2024-12
Statistics calculated
Number of nodes: 779
Number of edges: 247898
First network created
Pruning 222970 edges to reach average degree of 64...
Finished. Final Average Degree: 64.00
Threshold 0.7349 saved for Cluj_1960-01_2024-12
Network pruned and saved
Starting to work on Bacskatopolya : 1960-01 - 1990-12
Statistics calculated
Number of nodes: 371
Number of 

# Creating monthly networks

In [1]:
import sys
sys.path.append('../..')

import pandas as pd
import networkx as nx
import itertools

from networks.utils.get_monthly_nodes import get_monthly_nodes
# from networks.utils.calculate_similarity import calculate_similarity
from networks.utils.calculate_better_similarity import calculate_similarity
from networks.utils.prune_to_average_degree import prune_to_average_degree

from itertools import combinations
import numpy as np
from dtaidistance import dtw

In [2]:
def create_all_monthly_networks(specifications):
    for city, start, end in specifications:
        for month_index in range(1, 13):

            print(f"Starting to work on {city} : {start} - {end} | {month_index}")
            
            monthly_nodes = get_monthly_nodes(city, start, end, target_month=month_index)
            months = list(monthly_nodes.keys())

            dtw_distances = []

            for m1, m2 in combinations(months, 2):
                d1 = np.array(monthly_nodes[m1]['tg_derivatives'], dtype=np.double)
                d2 = np.array(monthly_nodes[m2]['tg_derivatives'], dtype=np.double)

                # z-normalize the sequences (important)
                d1 = (d1 - d1.mean()) / d1.std()
                d2 = (d2 - d2.mean()) / d2.std()

                dist = dtw.distance(d1, d2)
                dtw_distances.append(dist)

            dtw_mean = np.mean(dtw_distances)
            dtw_std  = np.std(dtw_distances)

            tg_diffs, tn_diffs, tx_diffs = [], [], []

            for m1, m2 in combinations(months, 2):
                tg_diffs.append(abs(monthly_nodes[m1]['mean_tg'] - monthly_nodes[m2]['mean_tg']))
                tn_diffs.append(abs(monthly_nodes[m1]['mean_tn'] - monthly_nodes[m2]['mean_tn']))
                tx_diffs.append(abs(monthly_nodes[m1]['mean_tx'] - monthly_nodes[m2]['mean_tx']))

            tg_mean, tg_std = np.mean(tg_diffs), np.std(tg_diffs)
            tn_mean, tn_std = np.mean(tn_diffs), np.std(tn_diffs)
            tx_mean, tx_std = np.mean(tx_diffs), np.std(tx_diffs)

            stats = {
                'dtw_mean': dtw_mean,
                'dtw_std': dtw_std,
                'tg_mean': tg_mean,
                'tg_std': tg_std,
                'tn_mean': tn_mean,
                'tn_std': tn_std,
                'tx_mean': tx_mean,
                'tx_std': tx_std
            }

            print(f"Statistics calculated")

            # --- Get the monthly data ---
            monthly_nodes = get_monthly_nodes(city, start, end, target_month=month_index)

            # --- Network Construction ---
            G = nx.Graph()
            simple_G = nx.Graph()

            # Add nodes to the graph
            for month, data in monthly_nodes.items():
                G.add_node(month, **data)
                simple_G.add_node(month)

            # --- Calculate similarities and add edges ---
            months = list(monthly_nodes.keys())
            similarity_threshold = 0.5

            for month1, month2 in itertools.combinations(months, 2):
                month1_data = monthly_nodes[month1]
                month2_data = monthly_nodes[month2]
                
                similarity_score = calculate_similarity(month1_data, month2_data, stats)
                
                if similarity_score > similarity_threshold:
                    G.add_edge(month1, month2, weight=similarity_score)
                    simple_G.add_edge(month1, month2, weight = similarity_score)

            # --- Inspect the graph ---
            print(f"Number of nodes: {G.number_of_nodes()}")
            print(f"Number of edges: {G.number_of_edges()}")

            print(f"First network created")

            g_type = 'monthly'

            graph_path = f"{city}_{g_type}_{month_index}_{start}_{end}_sim_{int(similarity_threshold*100)}.graphml"
            nx.write_graphml(simple_G, graph_path)

            # pruned_graph = prune_to_average_degree(graph_path, target_avg_degree=64)
            # nx.write_graphml(simple_G, graph_path)

            pruned_graph = simple_G

            # G = nx.read_graphml(graph_path)
            G = pruned_graph

            for node, data in G.nodes(data=True):
                try:
                    year, month = node.split("-")
                    data["year"] = int(year)
                    data["month"] = int(month)
                except ValueError:
                    data["year"] = None
                    data["month"] = None

            pruned_YM_path = f"{city}_{g_type}_{month_index}_pruned_{start}_{end}.graphml"

            nx.write_graphml(G, pruned_YM_path)

            print(f"Network pruned and saved")

In [3]:
specifications = [
    ('Cluj', '1960-01', '1990-12'),
    ('Cluj', '1991-01', '2024-12'),
    ('Bacskatopolya', '1960-01', '1990-12'),
    ('Bacskatopolya', '1991-01', '2024-12'),
    ('Brasov', '1960-01', '1990-12'),
    ('Brasov', '1991-01', '2024-12'),
    ('Deva', '1960-01', '1990-12'),
    ('Deva', '1991-01', '2024-12'),
    ('Gheorgheni', '1960-01', '1990-12'),
    ('Gheorgheni', '1991-01', '2024-12'),
    ('Gyor', '1960-01', '1990-12'),
    ('Gyor', '1991-01', '2024-12'),
    ('Kassa', '1960-01', '1990-12'),
    ('Kassa', '1991-01', '2024-12'),
    ('Kecskemet', '1960-01', '1990-12'),
    ('Kecskemet', '1991-01', '2024-12'),
    ('Keszthely', '1960-01', '1990-12'),
    ('Keszthely', '1991-01', '2024-12'),
    ('Oradea', '1960-01', '1990-12'),
    ('Oradea', '1991-01', '2024-12'),
    ('Pecs', '1960-01', '1990-12'),
    ('Pecs', '1991-01', '2024-12'),
]